In [45]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/divyanths/datasets-combined/cleaned_inpatient.csv
/kaggle/input/datasets/divyanths/datasets-combined/beneficiary_preprocessed.csv
/kaggle/input/datasets/divyanths/datasets-combined/cleaned_outpatient_data.csv
/kaggle/input/datasets/divyanths/datasets-combined/Train-1542865627584.csv


In [46]:
import pandas as pd
import numpy as np

In [47]:
beneficiary_df = pd.read_csv(
    "/kaggle/input/datasets/divyanths/datasets-combined/beneficiary_preprocessed.csv"
)

inpatient_df = pd.read_csv(
    "/kaggle/input/datasets/divyanths/datasets-combined/cleaned_inpatient.csv"
)

outpatient_df = pd.read_csv(
    "/kaggle/input/datasets/divyanths/datasets-combined/cleaned_outpatient_data.csv"
)

target_df = pd.read_csv(
    "/kaggle/input/datasets/divyanths/datasets-combined/Train-1542865627584.csv"
)

In [48]:
print("Beneficiary:", beneficiary_df.shape)
print("Inpatient:", inpatient_df.shape)
print("Outpatient:", outpatient_df.shape)
print("Target:", target_df.shape)

Beneficiary: (138556, 90)
Inpatient: (40474, 25)
Outpatient: (517737, 21)
Target: (5410, 2)


In [49]:
print("========== BENEFICIARY COLUMNS ==========")
print(beneficiary_df.columns.tolist())

print("\n========== INPATIENT COLUMNS ==========")
print(inpatient_df.columns.tolist())

print("\n========== OUTPATIENT COLUMNS ==========")
print(outpatient_df.columns.tolist())

print("\n========== TARGET COLUMNS ==========")
print(target_df.columns.tolist())

========== BENEFICIARY COLUMNS ==========
['BeneID', 'NoOfMonths_PartACov', 'NoOfMonths_PartBCov', 'ChronicCond_Alzheimer', 'ChronicCond_Heartfailure', 'ChronicCond_KidneyDisease', 'ChronicCond_Cancer', 'ChronicCond_ObstrPulmonary', 'ChronicCond_Depression', 'ChronicCond_Diabetes', 'ChronicCond_IschemicHeart', 'ChronicCond_Osteoporasis', 'ChronicCond_rheumatoidarthritis', 'ChronicCond_stroke', 'IPAnnualReimbursementAmt', 'IPAnnualDeductibleAmt', 'OPAnnualReimbursementAmt', 'OPAnnualDeductibleAmt', 'Age', 'ChronicConditionCount', 'TotalCoverageMonths', 'TotalReimbursement', 'TotalDeductible', 'TotalHealthcareCost', 'ReimbursementPerCoverageMonth', 'DeductiblePerCoverageMonth', 'ZeroCoverageFlag', 'DeceasedFlag', 'RenalDiseaseFlag', 'GenderFlag', 'Race_1', 'Race_2', 'Race_3', 'Race_5', 'State_1', 'State_2', 'State_3', 'State_4', 'State_5', 'State_6', 'State_7', 'State_8', 'State_9', 'State_10', 'State_11', 'State_12', 'State_13', 'State_14', 'State_15', 'State_16', 'State_17', 'State_18'

In [50]:
# Check BeneID mapping between Inpatient and Beneficiary

inpatient_bene_ids = set(inpatient_df["BeneID"].unique())
beneficiary_ids = set(beneficiary_df["BeneID"].unique())

matched_bene_ids = inpatient_bene_ids.intersection(beneficiary_ids)
unmatched_bene_ids = inpatient_bene_ids - beneficiary_ids

print("Total Inpatient BeneIDs:", len(inpatient_bene_ids))
print("Matched with Beneficiary:", len(matched_bene_ids))
print("Not found in Beneficiary:", len(unmatched_bene_ids))

Total Inpatient BeneIDs: 31289
Matched with Beneficiary: 31289
Not found in Beneficiary: 0


In [51]:
# Add Beneficiary information to Inpatient claims

inpatient_enriched = inpatient_df.merge(
    beneficiary_df,
    on="BeneID",
    how="left"
)

print("Original Inpatient shape:", inpatient_df.shape)
print("Enriched Inpatient shape:", inpatient_enriched.shape)

Original Inpatient shape: (40474, 25)
Enriched Inpatient shape: (40474, 114)


In [52]:
print("Missing BeneID:", inpatient_enriched["BeneID"].isna().sum())

print(
    "Rows with missing beneficiary features:",
    inpatient_enriched["Age"].isna().sum()
)

print(
    "Duplicate ClaimIDs:",
    inpatient_enriched["ClaimID"].duplicated().sum()
)

Missing BeneID: 0
Rows with missing beneficiary features: 0
Duplicate ClaimIDs: 0


In [53]:
# Check BeneID mapping between Outpatient and Beneficiary

outpatient_bene_ids = set(outpatient_df["BeneID"].unique())
beneficiary_ids = set(beneficiary_df["BeneID"].unique())

matched_outpatient_bene_ids = outpatient_bene_ids.intersection(beneficiary_ids)
unmatched_outpatient_bene_ids = outpatient_bene_ids - beneficiary_ids

print("Total Outpatient BeneIDs:", len(outpatient_bene_ids))
print("Matched with Beneficiary:", len(matched_outpatient_bene_ids))
print("Not found in Beneficiary:", len(unmatched_outpatient_bene_ids))

Total Outpatient BeneIDs: 133980
Matched with Beneficiary: 133980
Not found in Beneficiary: 0


In [54]:
# Add Beneficiary information to Outpatient claims

outpatient_enriched = outpatient_df.merge(
    beneficiary_df,
    on="BeneID",
    how="left"
)

print("Original Outpatient shape:", outpatient_df.shape)
print("Enriched Outpatient shape:", outpatient_enriched.shape)

Original Outpatient shape: (517737, 21)
Enriched Outpatient shape: (517737, 110)


In [55]:
print("Missing BeneID:", outpatient_enriched["BeneID"].isna().sum())

print(
    "Rows with missing beneficiary features:",
    outpatient_enriched["Age"].isna().sum()
)

print(
    "Duplicate ClaimIDs:",
    outpatient_enriched["ClaimID"].duplicated().sum()
)

Missing BeneID: 0
Rows with missing beneficiary features: 0
Duplicate ClaimIDs: 0


In [56]:
# Check Provider mapping for Inpatient

inpatient_providers = set(inpatient_enriched["Provider"].unique())
target_providers = set(target_df["Provider"].unique())

inpatient_matched = inpatient_providers.intersection(target_providers)
inpatient_unmatched = inpatient_providers - target_providers

print("========== INPATIENT PROVIDER MAPPING ==========")
print("Inpatient Providers:", len(inpatient_providers))
print("Matched with Target:", len(inpatient_matched))
print("Not found in Target:", len(inpatient_unmatched))


# Check Provider mapping for Outpatient

outpatient_providers = set(outpatient_enriched["Provider"].unique())

outpatient_matched = outpatient_providers.intersection(target_providers)
outpatient_unmatched = outpatient_providers - target_providers

print("\n========== OUTPATIENT PROVIDER MAPPING ==========")
print("Outpatient Providers:", len(outpatient_providers))
print("Matched with Target:", len(outpatient_matched))
print("Not found in Target:", len(outpatient_unmatched))

========== INPATIENT PROVIDER MAPPING ==========
Inpatient Providers: 2092
Matched with Target: 2092
Not found in Target: 0

========== OUTPATIENT PROVIDER MAPPING ==========
Outpatient Providers: 5012
Matched with Target: 5012
Not found in Target: 0


In [57]:
inpatient_enriched = inpatient_enriched.merge(
    target_df[["Provider", "PotentialFraud"]],
    on="Provider",
    how="left"
)

print("Inpatient shape:", inpatient_enriched.shape)
print(
    "Missing PotentialFraud:",
    inpatient_enriched["PotentialFraud"].isna().sum()
)

Inpatient shape: (40474, 115)
Missing PotentialFraud: 0


In [58]:
outpatient_enriched = outpatient_enriched.merge(
    target_df[["Provider", "PotentialFraud"]],
    on="Provider",
    how="left"
)

print("Outpatient shape:", outpatient_enriched.shape)
print(
    "Missing PotentialFraud:",
    outpatient_enriched["PotentialFraud"].isna().sum()
)

Outpatient shape: (517737, 111)
Missing PotentialFraud: 0


In [59]:
print("========== INPATIENT DATA TYPES ==========")

print("\nNumeric columns:")
print(inpatient_enriched.select_dtypes(include=np.number).columns.tolist())

print("\nObject/Categorical columns:")
print(inpatient_enriched.select_dtypes(include="object").columns.tolist())

========== INPATIENT DATA TYPES ==========

Numeric columns:
['InscClaimAmtReimbursed', 'ClmProcedureCode_1', 'ClmProcedureCode_2', 'NoOfMonths_PartACov', 'NoOfMonths_PartBCov', 'ChronicCond_Alzheimer', 'ChronicCond_Heartfailure', 'ChronicCond_KidneyDisease', 'ChronicCond_Cancer', 'ChronicCond_ObstrPulmonary', 'ChronicCond_Depression', 'ChronicCond_Diabetes', 'ChronicCond_IschemicHeart', 'ChronicCond_Osteoporasis', 'ChronicCond_rheumatoidarthritis', 'ChronicCond_stroke', 'IPAnnualReimbursementAmt', 'IPAnnualDeductibleAmt', 'OPAnnualReimbursementAmt', 'OPAnnualDeductibleAmt', 'Age', 'ChronicConditionCount', 'TotalCoverageMonths', 'TotalReimbursement', 'TotalDeductible', 'TotalHealthcareCost', 'ReimbursementPerCoverageMonth', 'DeductiblePerCoverageMonth', 'ZeroCoverageFlag', 'DeceasedFlag', 'RenalDiseaseFlag', 'GenderFlag', 'Race_1', 'Race_2', 'Race_3', 'Race_5', 'State_1', 'State_2', 'State_3', 'State_4', 'State_5', 'State_6', 'State_7', 'State_8', 'State_9', 'State_10', 'State_11', 'St

In [60]:
print("\n========== OUTPATIENT DATA TYPES ==========")

print("\nNumeric columns:")
print(outpatient_enriched.select_dtypes(include=np.number).columns.tolist())

print("\nObject/Categorical columns:")
print(outpatient_enriched.select_dtypes(include="object").columns.tolist())


========== OUTPATIENT DATA TYPES ==========

Numeric columns:
['InscClaimAmtReimbursed', 'DeductibleAmtPaid', 'NoOfMonths_PartACov', 'NoOfMonths_PartBCov', 'ChronicCond_Alzheimer', 'ChronicCond_Heartfailure', 'ChronicCond_KidneyDisease', 'ChronicCond_Cancer', 'ChronicCond_ObstrPulmonary', 'ChronicCond_Depression', 'ChronicCond_Diabetes', 'ChronicCond_IschemicHeart', 'ChronicCond_Osteoporasis', 'ChronicCond_rheumatoidarthritis', 'ChronicCond_stroke', 'IPAnnualReimbursementAmt', 'IPAnnualDeductibleAmt', 'OPAnnualReimbursementAmt', 'OPAnnualDeductibleAmt', 'Age', 'ChronicConditionCount', 'TotalCoverageMonths', 'TotalReimbursement', 'TotalDeductible', 'TotalHealthcareCost', 'ReimbursementPerCoverageMonth', 'DeductiblePerCoverageMonth', 'ZeroCoverageFlag', 'DeceasedFlag', 'RenalDiseaseFlag', 'GenderFlag', 'Race_1', 'Race_2', 'Race_3', 'Race_5', 'State_1', 'State_2', 'State_3', 'State_4', 'State_5', 'State_6', 'State_7', 'State_8', 'State_9', 'State_10', 'State_11', 'State_12', 'State_13', 

In [61]:
inpatient_provider_bene = (
    inpatient_enriched[
        ["Provider", "BeneID"]
    ]
    .drop_duplicates()
)

print("Unique Provider-BeneID pairs:",
      len(inpatient_provider_bene))

print("\nSample:")
print(inpatient_provider_bene.head())

Unique Provider-BeneID pairs: 36616

Sample:
   Provider     BeneID
0  PRV55912  BENE11001
1  PRV55907  BENE11001
2  PRV56046  BENE11001
3  PRV52405  BENE11011
4  PRV56614  BENE11014


In [62]:
inpatient_provider_bene_features = inpatient_provider_bene.merge(
    beneficiary_df,
    on="BeneID",
    how="left"
)

print("Shape:", inpatient_provider_bene_features.shape)
print("Missing BeneID:",
      inpatient_provider_bene_features["BeneID"].isna().sum())
print("Missing Age:",
      inpatient_provider_bene_features["Age"].isna().sum())

Shape: (36616, 91)
Missing BeneID: 0
Missing Age: 0


In [63]:
beneficiary_feature_cols = [
    col for col in beneficiary_df.columns
    if col != "BeneID"
]

print("Number of beneficiary features:",
      len(beneficiary_feature_cols))

Number of beneficiary features: 89


In [64]:
inpatient_beneficiary_features = (
    inpatient_provider_bene_features
    .groupby("Provider")[beneficiary_feature_cols]
    .mean()
    .reset_index()
)

print("Shape:", inpatient_beneficiary_features.shape)
print(inpatient_beneficiary_features.head())

Shape: (2092, 90)
   Provider  NoOfMonths_PartACov  NoOfMonths_PartBCov  ChronicCond_Alzheimer  \
0  PRV51001            12.000000            12.000000               1.600000   
1  PRV51003            11.773585            11.773585               1.566038   
2  PRV51007            12.000000            12.000000               1.333333   
3  PRV51008            12.000000            12.000000               1.500000   
4  PRV51011            12.000000            12.000000               1.000000   

   ChronicCond_Heartfailure  ChronicCond_KidneyDisease  ChronicCond_Cancer  \
0                  1.200000                   1.200000            1.800000   
1                  1.433962                   1.415094            1.867925   
2                  1.000000                   1.666667            2.000000   
3                  2.000000                   1.500000            1.500000   
4                  2.000000                   1.000000            2.000000   

   ChronicCond_ObstrPulmonary  C

In [65]:
chronic_cols = [
    col for col in beneficiary_df.columns
    if col.startswith("ChronicCond_")
]

for col in chronic_cols:
    print(col, ":", sorted(beneficiary_df[col].dropna().unique()))

ChronicCond_Alzheimer : [np.int64(1), np.int64(2)]
ChronicCond_Heartfailure : [np.int64(1), np.int64(2)]
ChronicCond_KidneyDisease : [np.int64(1), np.int64(2)]
ChronicCond_Cancer : [np.int64(1), np.int64(2)]
ChronicCond_ObstrPulmonary : [np.int64(1), np.int64(2)]
ChronicCond_Depression : [np.int64(1), np.int64(2)]
ChronicCond_Diabetes : [np.int64(1), np.int64(2)]
ChronicCond_IschemicHeart : [np.int64(1), np.int64(2)]
ChronicCond_Osteoporasis : [np.int64(1), np.int64(2)]
ChronicCond_rheumatoidarthritis : [np.int64(1), np.int64(2)]
ChronicCond_stroke : [np.int64(1), np.int64(2)]


In [66]:
chronic_cols = [
    col for col in beneficiary_df.columns
    if col.startswith("ChronicCond_")
]

for col in chronic_cols:
    beneficiary_df[col] = (beneficiary_df[col] == 1).astype(int)

In [67]:
chronic_cols = [
    col for col in beneficiary_df.columns
    if col.startswith("ChronicCond_")
]

for col in chronic_cols:
    beneficiary_df[col] = (beneficiary_df[col] == 1).astype(int)

print("Chronic condition columns converted successfully.")

for col in chronic_cols:
    print(col, ":", sorted(beneficiary_df[col].unique()))

Chronic condition columns converted successfully.
ChronicCond_Alzheimer : [np.int64(0), np.int64(1)]
ChronicCond_Heartfailure : [np.int64(0), np.int64(1)]
ChronicCond_KidneyDisease : [np.int64(0), np.int64(1)]
ChronicCond_Cancer : [np.int64(0), np.int64(1)]
ChronicCond_ObstrPulmonary : [np.int64(0), np.int64(1)]
ChronicCond_Depression : [np.int64(0), np.int64(1)]
ChronicCond_Diabetes : [np.int64(0), np.int64(1)]
ChronicCond_IschemicHeart : [np.int64(0), np.int64(1)]
ChronicCond_Osteoporasis : [np.int64(0), np.int64(1)]
ChronicCond_rheumatoidarthritis : [np.int64(0), np.int64(1)]
ChronicCond_stroke : [np.int64(0), np.int64(1)]


In [68]:
# Recreate the Provider-BeneID mapping
inpatient_provider_bene_features = inpatient_provider_bene.merge(
    beneficiary_df,
    on="BeneID",
    how="left"
)

# Get all beneficiary features except BeneID
beneficiary_feature_cols = [
    col for col in beneficiary_df.columns
    if col != "BeneID"
]

# Aggregate beneficiary information at Provider level
inpatient_beneficiary_features = (
    inpatient_provider_bene_features
    .groupby("Provider")[beneficiary_feature_cols]
    .mean()
    .reset_index()
)

print("Shape:", inpatient_beneficiary_features.shape)
print(inpatient_beneficiary_features.head())

Shape: (2092, 90)
   Provider  NoOfMonths_PartACov  NoOfMonths_PartBCov  ChronicCond_Alzheimer  \
0  PRV51001            12.000000            12.000000               0.400000   
1  PRV51003            11.773585            11.773585               0.433962   
2  PRV51007            12.000000            12.000000               0.666667   
3  PRV51008            12.000000            12.000000               0.500000   
4  PRV51011            12.000000            12.000000               1.000000   

   ChronicCond_Heartfailure  ChronicCond_KidneyDisease  ChronicCond_Cancer  \
0                  0.800000                   0.800000            0.200000   
1                  0.566038                   0.584906            0.132075   
2                  1.000000                   0.333333            0.000000   
3                  0.000000                   0.500000            0.500000   
4                  0.000000                   1.000000            0.000000   

   ChronicCond_ObstrPulmonary  C

In [69]:
inpatient_claim_features = (
    inpatient_enriched
    .groupby("Provider")
    .agg(
        InpatientClaimCount=("ClaimID", "nunique"),
        InpatientBeneficiaryCount=("BeneID", "nunique"),
        InpatientTotalReimbursement=("InscClaimAmtReimbursed", "sum"),
        InpatientAvgReimbursement=("InscClaimAmtReimbursed", "mean"),
        InpatientMaxReimbursement=("InscClaimAmtReimbursed", "max"),
        InpatientMinReimbursement=("InscClaimAmtReimbursed", "min")
    )
    .reset_index()
)

print("Shape:", inpatient_claim_features.shape)
print(inpatient_claim_features.head())

Shape: (2092, 7)
   Provider  InpatientClaimCount  InpatientBeneficiaryCount  \
0  PRV51001                    5                          5   
1  PRV51003                   62                         53   
2  PRV51007                    3                          3   
3  PRV51008                    2                          2   
4  PRV51011                    1                          1   

   InpatientTotalReimbursement  InpatientAvgReimbursement  \
0                        97000               19400.000000   
1                       573000                9241.935484   
2                        19000                6333.333333   
3                        25000               12500.000000   
4                         5000                5000.000000   

   InpatientMaxReimbursement  InpatientMinReimbursement  
0                      42000                       3000  
1                      57000                          0  
2                      10000                       3000  
3    

In [70]:
inpatient_provider_features = inpatient_beneficiary_features.merge(
    inpatient_claim_features,
    on="Provider",
    how="left"
)

print("Inpatient Provider Features Shape:",
      inpatient_provider_features.shape)

print("\nDuplicate Providers:",
      inpatient_provider_features["Provider"].duplicated().sum())

print("\nMissing values:")
print(
    inpatient_provider_features.isna().sum()
    .sort_values(ascending=False)
    .head(10)
)

Inpatient Provider Features Shape: (2092, 96)

Duplicate Providers: 0

Missing values:
Provider                      0
NoOfMonths_PartACov           0
NoOfMonths_PartBCov           0
ChronicCond_Alzheimer         0
ChronicCond_Heartfailure      0
ChronicCond_KidneyDisease     0
ChronicCond_Cancer            0
ChronicCond_ObstrPulmonary    0
ChronicCond_Depression        0
ChronicCond_Diabetes          0
dtype: int64


In [71]:
outpatient_provider_bene = (
    outpatient_enriched[
        ["Provider", "BeneID"]
    ]
    .drop_duplicates()
)

print(
    "Unique Provider-BeneID pairs:",
    len(outpatient_provider_bene)
)

print("\nSample:")
print(outpatient_provider_bene.head())

Unique Provider-BeneID pairs: 330821

Sample:
    Provider     BeneID
0   PRV56011  BENE11002
1   PRV57610  BENE11003
2   PRV57595  BENE11003
3   PRV56011  BENE11004
10  PRV55951  BENE11004


In [72]:
outpatient_provider_bene_features = outpatient_provider_bene.merge(
    beneficiary_df,
    on="BeneID",
    how="left"
)

print("Shape:", outpatient_provider_bene_features.shape)

print("Missing BeneID:",
      outpatient_provider_bene_features["BeneID"].isna().sum())

print("Missing Age:",
      outpatient_provider_bene_features["Age"].isna().sum())

Shape: (330821, 91)
Missing BeneID: 0
Missing Age: 0


In [73]:
outpatient_beneficiary_features = (
    outpatient_provider_bene_features
    .groupby("Provider")[beneficiary_feature_cols]
    .mean()
    .reset_index()
)

print("Shape:", outpatient_beneficiary_features.shape)
print(outpatient_beneficiary_features.head())

Shape: (5012, 90)
   Provider  NoOfMonths_PartACov  NoOfMonths_PartBCov  ChronicCond_Alzheimer  \
0  PRV51001            12.000000            12.000000               0.631579   
1  PRV51003            11.818182            11.924242               0.348485   
2  PRV51004            11.855072            11.956522               0.434783   
3  PRV51005            11.830303            11.886869               0.333333   
4  PRV51007            11.785714            11.785714               0.357143   

   ChronicCond_Heartfailure  ChronicCond_KidneyDisease  ChronicCond_Cancer  \
0                  0.736842                   0.684211            0.210526   
1                  0.636364                   0.348485            0.045455   
2                  0.594203                   0.340580            0.115942   
3                  0.531313                   0.359596            0.119192   
4                  0.500000                   0.303571            0.107143   

   ChronicCond_ObstrPulmonary  C

In [74]:
outpatient_claim_features = (
    outpatient_enriched
    .groupby("Provider")
    .agg(
        OutpatientClaimCount=("ClaimID", "nunique"),
        OutpatientBeneficiaryCount=("BeneID", "nunique"),
        OutpatientTotalReimbursement=("InscClaimAmtReimbursed", "sum"),
        OutpatientAvgReimbursement=("InscClaimAmtReimbursed", "mean"),
        OutpatientMaxReimbursement=("InscClaimAmtReimbursed", "max"),
        OutpatientMinReimbursement=("InscClaimAmtReimbursed", "min"),
        OutpatientTotalDeductible=("DeductibleAmtPaid", "sum"),
        OutpatientAvgDeductible=("DeductibleAmtPaid", "mean"),
        OutpatientMaxDeductible=("DeductibleAmtPaid", "max"),
        OutpatientMinDeductible=("DeductibleAmtPaid", "min")
    )
    .reset_index()
)

print("Shape:", outpatient_claim_features.shape)
print(outpatient_claim_features.head())

Shape: (5012, 11)
   Provider  OutpatientClaimCount  OutpatientBeneficiaryCount  \
0  PRV51001                    20                          19   
1  PRV51003                    70                          66   
2  PRV51004                   149                         138   
3  PRV51005                  1165                         495   
4  PRV51007                    69                          56   

   OutpatientTotalReimbursement  OutpatientAvgReimbursement  \
0                          7640                  382.000000   
1                         32670                  466.714286   
2                         52170                  350.134228   
3                        280910                  241.124464   
4                         14710                  213.188406   

   OutpatientMaxReimbursement  OutpatientMinReimbursement  \
0                        1500                          10   
1                        3300                           0   
2                        3300

In [75]:
outpatient_provider_features = outpatient_beneficiary_features.merge(
    outpatient_claim_features,
    on="Provider",
    how="left"
)

print(
    "Outpatient Provider Features Shape:",
    outpatient_provider_features.shape
)

print(
    "\nDuplicate Providers:",
    outpatient_provider_features["Provider"].duplicated().sum()
)

print("\nMissing values:")
print(
    outpatient_provider_features.isna().sum()
    .sort_values(ascending=False)
    .head(10)
)

Outpatient Provider Features Shape: (5012, 100)

Duplicate Providers: 0

Missing values:
Provider                      0
NoOfMonths_PartACov           0
NoOfMonths_PartBCov           0
ChronicCond_Alzheimer         0
ChronicCond_Heartfailure      0
ChronicCond_KidneyDisease     0
ChronicCond_Cancer            0
ChronicCond_ObstrPulmonary    0
ChronicCond_Depression        0
ChronicCond_Diabetes          0
dtype: int64


In [76]:
final_df = target_df.merge(
    inpatient_provider_features,
    on="Provider",
    how="left"
)

final_df = final_df.merge(
    outpatient_provider_features,
    on="Provider",
    how="left"
)

print("Final Dataset Shape:", final_df.shape)
print("Unique Providers:", final_df["Provider"].nunique())
print("Duplicate Providers:", final_df["Provider"].duplicated().sum())

Final Dataset Shape: (5410, 196)
Unique Providers: 5410
Duplicate Providers: 0


In [77]:
print("========== FINAL DATASET CHECK ==========")

print("Shape:", final_df.shape)

print("Unique Providers:", final_df["Provider"].nunique())

print("Duplicate Providers:",
      final_df["Provider"].duplicated().sum())

print("\nPotentialFraud distribution:")
print(final_df["PotentialFraud"].value_counts())

print("\nMissing values:")
print(
    final_df.isna().sum()
    .sort_values(ascending=False)
    .head(20)
)

========== FINAL DATASET CHECK ==========
Shape: (5410, 196)
Unique Providers: 5410
Duplicate Providers: 0

PotentialFraud distribution:
PotentialFraud
No     4904
Yes     506
Name: count, dtype: int64

Missing values:
NoOfMonths_PartBCov_x                3318
NoOfMonths_PartACov_x                3318
ChronicCond_Heartfailure_x           3318
ChronicCond_Alzheimer_x              3318
State_2_x                            3318
State_3_x                            3318
ChronicCond_KidneyDisease_x          3318
ChronicCond_Cancer_x                 3318
ChronicCond_ObstrPulmonary_x         3318
ChronicCond_Depression_x             3318
ChronicCond_Diabetes_x               3318
ChronicCond_IschemicHeart_x          3318
ChronicCond_Osteoporasis_x           3318
ChronicCond_rheumatoidarthritis_x    3318
ChronicCond_stroke_x                 3318
IPAnnualReimbursementAmt_x           3318
IPAnnualDeductibleAmt_x              3318
OPAnnualReimbursementAmt_x           3318
OPAnnualDeductibleAmt_x  

In [78]:
# Check how many providers have no Inpatient data
inpatient_missing = final_df[
    "InpatientClaimCount"
].isna().sum()

# Check how many providers have no Outpatient data
outpatient_missing = final_df[
    "OutpatientClaimCount"
].isna().sum()

print("Providers with no Inpatient data:", inpatient_missing)
print("Providers with no Outpatient data:", outpatient_missing)

Providers with no Inpatient data: 3318
Providers with no Outpatient data: 398


In [79]:
inpatient_claim_cols = [
    "InpatientClaimCount",
    "InpatientBeneficiaryCount",
    "InpatientTotalReimbursement",
    "InpatientAvgReimbursement",
    "InpatientMaxReimbursement",
    "InpatientMinReimbursement"
]

outpatient_claim_cols = [
    "OutpatientClaimCount",
    "OutpatientBeneficiaryCount",
    "OutpatientTotalReimbursement",
    "OutpatientAvgReimbursement",
    "OutpatientMaxReimbursement",
    "OutpatientMinReimbursement",
    "OutpatientTotalDeductible",
    "OutpatientAvgDeductible",
    "OutpatientMaxDeductible",
    "OutpatientMinDeductible"
]

for col in inpatient_claim_cols:
    final_df[col] = final_df[col].fillna(0)

for col in outpatient_claim_cols:
    final_df[col] = final_df[col].fillna(0)

print("Missing Inpatient claim features:",
      final_df[inpatient_claim_cols].isna().sum().sum())

print("Missing Outpatient claim features:",
      final_df[outpatient_claim_cols].isna().sum().sum())

Missing Inpatient claim features: 0
Missing Outpatient claim features: 0


In [80]:
remaining_missing = (
    final_df.isna()
    .sum()
    .sort_values(ascending=False)
)

print(remaining_missing[remaining_missing > 0])

NoOfMonths_PartBCov_x         3318
NoOfMonths_PartACov_x         3318
ChronicCond_Heartfailure_x    3318
ChronicCond_Alzheimer_x       3318
State_2_x                     3318
                              ... 
State_50_y                     398
State_46_y                     398
State_47_y                     398
State_51_y                     398
State_52_y                     398
Length: 178, dtype: int64


In [81]:
# Rename Inpatient and Outpatient beneficiary feature columns

rename_dict = {}

for col in final_df.columns:
    if col.endswith("_x"):
        rename_dict[col] = col[:-2] + "_Inpatient"
    elif col.endswith("_y"):
        rename_dict[col] = col[:-2] + "_Outpatient"

final_df = final_df.rename(columns=rename_dict)

print("Columns renamed successfully.")

print("\nSample renamed columns:")
print([
    col for col in final_df.columns
    if "Age" in col or "ChronicCond_Diabetes" in col
])

Columns renamed successfully.

Sample renamed columns:
['ChronicCond_Diabetes_Inpatient', 'Age_Inpatient', 'ChronicCond_Diabetes_Outpatient', 'Age_Outpatient']


In [82]:
# Fill missing provider-side beneficiary features
# Missing means the provider has no claims/beneficiaries
# from that particular dataset.

beneficiary_inpatient_cols = [
    col for col in final_df.columns
    if col.endswith("_Inpatient")
]

beneficiary_outpatient_cols = [
    col for col in final_df.columns
    if col.endswith("_Outpatient")
]

for col in beneficiary_inpatient_cols:
    final_df[col] = final_df[col].fillna(0)

for col in beneficiary_outpatient_cols:
    final_df[col] = final_df[col].fillna(0)

print(
    "Remaining missing values:",
    final_df.isna().sum().sum()
)

Remaining missing values: 0


In [83]:
print("========== FINAL DATASET VALIDATION ==========")

print("Shape:", final_df.shape)

print("Unique Providers:",
      final_df["Provider"].nunique())

print("Duplicate Providers:",
      final_df["Provider"].duplicated().sum())

print("\nMissing values:",
      final_df.isna().sum().sum())

print("\nPotentialFraud distribution:")
print(final_df["PotentialFraud"].value_counts())

print("\nPotentialFraud percentage:")
print(
    final_df["PotentialFraud"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

========== FINAL DATASET VALIDATION ==========
Shape: (5410, 196)
Unique Providers: 5410
Duplicate Providers: 0

Missing values: 0

PotentialFraud distribution:
PotentialFraud
No     4904
Yes     506
Name: count, dtype: int64

PotentialFraud percentage:
PotentialFraud
No     90.65
Yes     9.35
Name: proportion, dtype: float64


In [84]:
# Move PotentialFraud to the last column
cols = [col for col in final_df.columns if col != "PotentialFraud"]
cols.append("PotentialFraud")

final_df = final_df[cols]

print("Last 5 columns:")
print(final_df.columns[-5:])

Last 5 columns:
Index(['OutpatientTotalDeductible', 'OutpatientAvgDeductible',
       'OutpatientMaxDeductible', 'OutpatientMinDeductible', 'PotentialFraud'],
      dtype='object')


In [85]:
# Create copies so the original enriched datasets are not changed

inpatient_cases = inpatient_enriched.copy()
outpatient_cases = outpatient_enriched.copy()

# Identify the type of claim
inpatient_cases["ClaimType"] = "Inpatient"
outpatient_cases["ClaimType"] = "Outpatient"

print("Inpatient cases:", inpatient_cases.shape)
print("Outpatient cases:", outpatient_cases.shape)

Inpatient cases: (40474, 116)
Outpatient cases: (517737, 112)


In [86]:
chronic_cols = [
    col for col in beneficiary_df.columns
    if col.startswith("ChronicCond_")
]

for col in chronic_cols:
    beneficiary_df[col] = (beneficiary_df[col] == 1).astype(int)

print("Chronic condition conversion completed.")

Chronic condition conversion completed.


In [87]:
inpatient_enriched = inpatient_df.merge(
    beneficiary_df,
    on="BeneID",
    how="left"
)

inpatient_enriched = inpatient_enriched.merge(
    target_df[["Provider", "PotentialFraud"]],
    on="Provider",
    how="left"
)

print("Inpatient enriched shape:", inpatient_enriched.shape)
print(
    "Missing PotentialFraud:",
    inpatient_enriched["PotentialFraud"].isna().sum()
)

Inpatient enriched shape: (40474, 115)
Missing PotentialFraud: 0


In [89]:
outpatient_enriched = outpatient_df.merge(
    beneficiary_df,
    on="BeneID",
    how="left"
)

outpatient_enriched = outpatient_enriched.merge(
    target_df[["Provider", "PotentialFraud"]],
    on="Provider",
    how="left"
)

print("Outpatient enriched shape:", outpatient_enriched.shape)
print(
    "Missing PotentialFraud:",
    outpatient_enriched["PotentialFraud"].isna().sum()
)

Outpatient enriched shape: (517737, 111)
Missing PotentialFraud: 0


In [90]:
# Make copies so the enriched datasets remain unchanged

inpatient_cases = inpatient_enriched.copy()
outpatient_cases = outpatient_enriched.copy()

# Identify the source of each claim
inpatient_cases["ClaimType"] = "Inpatient"
outpatient_cases["ClaimType"] = "Outpatient"

print("Inpatient cases:", inpatient_cases.shape)
print("Outpatient cases:", outpatient_cases.shape)

Inpatient cases: (40474, 116)
Outpatient cases: (517737, 112)


In [91]:
common_cols = sorted(
    set(inpatient_cases.columns)
    .intersection(outpatient_cases.columns)
)

inpatient_only_cols = sorted(
    set(inpatient_cases.columns)
    - set(outpatient_cases.columns)
)

outpatient_only_cols = sorted(
    set(outpatient_cases.columns)
    - set(inpatient_cases.columns)
)

print("Common columns:", len(common_cols))
print("Inpatient-only columns:", len(inpatient_only_cols))
print("Outpatient-only columns:", len(outpatient_only_cols))

print("\nInpatient-only columns:")
print(inpatient_only_cols)

print("\nOutpatient-only columns:")
print(outpatient_only_cols)

Common columns: 111
Inpatient-only columns: 5
Outpatient-only columns: 1

Inpatient-only columns:
['AdmissionDt', 'ClmProcedureCode_1', 'ClmProcedureCode_2', 'DiagnosisGroupCode', 'DischargeDt']

Outpatient-only columns:
['DeductibleAmtPaid']


In [100]:
# Combine Inpatient and Outpatient claim-level datasets
all_cases_df = pd.concat(
    [inpatient_cases, outpatient_cases],
    ignore_index=True,
    sort=False
)

print("Combined dataset shape:", all_cases_df.shape)
print("\nClaimType distribution:")
print(all_cases_df["ClaimType"].value_counts())

Combined dataset shape: (558211, 117)

ClaimType distribution:
ClaimType
Outpatient    517737
Inpatient      40474
Name: count, dtype: int64


In [101]:
print("========== KEY COLUMN CHECK ==========")

for col in ["BeneID", "Provider", "ClaimID", "ClaimType", "PotentialFraud"]:
    print(
        f"{col}:",
        "Present" if col in all_cases_df.columns else "MISSING"
    )

print("\nUnique BeneIDs:", all_cases_df["BeneID"].nunique())
print("Unique Providers:", all_cases_df["Provider"].nunique())
print("Unique ClaimIDs:", all_cases_df["ClaimID"].nunique())

print("\nMissing key values:")
print(
    all_cases_df[
        ["BeneID", "Provider", "ClaimID", "ClaimType", "PotentialFraud"]
    ].isna().sum()
)

========== KEY COLUMN CHECK ==========
BeneID: Present
Provider: Present
ClaimID: Present
ClaimType: Present
PotentialFraud: Present

Unique BeneIDs: 138556
Unique Providers: 5410
Unique ClaimIDs: 558211

Missing key values:
BeneID            0
Provider          0
ClaimID           0
ClaimType         0
PotentialFraud    0
dtype: int64


In [102]:
# Move PotentialFraud to the last column

cols = [
    col for col in all_cases_df.columns
    if col != "PotentialFraud"
]

cols.append("PotentialFraud")

all_cases_df = all_cases_df[cols]

print("Last 5 columns:")
print(all_cases_df.columns[-5:])

Last 5 columns:
Index(['ReimbursementPerCoverageMonth_Missing',
       'DeductiblePerCoverageMonth_Missing', 'ClaimType', 'DeductibleAmtPaid',
       'PotentialFraud'],
      dtype='object')


In [105]:
print("Final shape:", all_cases_df.shape)
print("First 10 columns:")
print(all_cases_df.columns[:10].tolist())

print("\nLast 10 columns:")
print(all_cases_df.columns[-10:].tolist())

Final shape: (558211, 117)
First 10 columns:
['BeneID', 'ClaimID', 'ClaimStartDt', 'ClaimEndDt', 'Provider', 'InscClaimAmtReimbursed', 'AttendingPhysician', 'OperatingPhysician', 'OtherPhysician', 'AdmissionDt']

Last 10 columns:
['State_52', 'State_53', 'State_54', 'CountyFrequency', 'CountyRarity', 'ReimbursementPerCoverageMonth_Missing', 'DeductiblePerCoverageMonth_Missing', 'ClaimType', 'DeductibleAmtPaid', 'PotentialFraud']


In [106]:
# Check missing values by ClaimType

print("========== MISSING VALUES BY CLAIM TYPE ==========")

inpatient_rows = all_cases_df["ClaimType"] == "Inpatient"
outpatient_rows = all_cases_df["ClaimType"] == "Outpatient"

print("\nInpatient rows - missing values:")
print(
    all_cases_df.loc[inpatient_rows].isna().sum()
    .sort_values(ascending=False)
    .head(10)
)

print("\nOutpatient rows - missing values:")
print(
    all_cases_df.loc[outpatient_rows].isna().sum()
    .sort_values(ascending=False)
    .head(10)
)

========== MISSING VALUES BY CLAIM TYPE ==========

Inpatient rows - missing values:
DeductibleAmtPaid         40474
ClmProcedureCode_2        35020
ClmProcedureCode_1        17326
ClaimID                       0
ClaimStartDt                  0
InscClaimAmtReimbursed        0
AttendingPhysician            0
OperatingPhysician            0
OtherPhysician                0
AdmissionDt                   0
dtype: int64

Outpatient rows - missing values:
DischargeDt               517737
DiagnosisGroupCode        517737
AdmissionDt               517737
ClmProcedureCode_1        517737
ClmProcedureCode_2        517737
Provider                       0
ClaimEndDt                     0
ClaimStartDt                   0
InscClaimAmtReimbursed         0
OtherPhysician                 0
dtype: int64


In [108]:
print("========== FINAL SINGLE DATASET CHECK ==========")

print("Shape:", all_cases_df.shape)

print("Unique BeneIDs:", all_cases_df["BeneID"].nunique())
print("Unique Providers:", all_cases_df["Provider"].nunique())
print("Unique ClaimIDs:", all_cases_df["ClaimID"].nunique())

print("\nDuplicate ClaimIDs:",
      all_cases_df["ClaimID"].duplicated().sum())

print("\nMissing key columns:")
print(
    all_cases_df[
        ["BeneID", "Provider", "ClaimID", "ClaimType", "PotentialFraud"]
    ].isna().sum()
)

print("\nClaimType:")
print(all_cases_df["ClaimType"].value_counts())

print("\nPotentialFraud:")
print(all_cases_df["PotentialFraud"].value_counts())

========== FINAL SINGLE DATASET CHECK ==========
Shape: (558211, 117)
Unique BeneIDs: 138556
Unique Providers: 5410
Unique ClaimIDs: 558211

Duplicate ClaimIDs: 0

Missing key columns:
BeneID            0
Provider          0
ClaimID           0
ClaimType         0
PotentialFraud    0
dtype: int64

ClaimType:
ClaimType
Outpatient    517737
Inpatient      40474
Name: count, dtype: int64

PotentialFraud:
PotentialFraud
No     345415
Yes    212796
Name: count, dtype: int64


In [109]:
all_cases_df.to_csv(
    "/kaggle/working/All Datasets Combined.csv",
    index=False
)

print("All Datasets Combined.csv created successfully!")

All Datasets Combined.csv created successfully!


In [110]:
import os

file_path = "/kaggle/working/All Datasets Combined.csv"

print("File exists:", os.path.exists(file_path))
print(
    "File size (MB):",
    round(os.path.getsize(file_path) / (1024 * 1024), 2)
)

print("\nLast column:", all_cases_df.columns[-1])

File exists: True
File size (MB): 227.57

Last column: PotentialFraud
